# Chunking Benchmark

This notebook compares `chunk_fixed`, `chunk_sentence`, and `chunk_semantic` on 100 contract descriptions.

- Real data path: load descriptions from PostgreSQL when `DATABASE_URL` is available and the connection succeeds.
- Safe fallback path: generate 100 deterministic sample descriptions when PostgreSQL is unavailable.
- Evaluation path: build 10 anchor queries from the loaded descriptions and score precision@5 / recall@5 against the anchor document's chunks.


In [ ]:
from __future__ import annotations

import asyncio
import hashlib
import math
import os
import re
import statistics
import sys
import threading
import time
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from govintel.ingestion.embedder import (
    chunk_fixed as project_chunk_fixed,
    chunk_semantic as project_chunk_semantic,
    chunk_sentence as project_chunk_sentence,
)

WORD_RE = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")
STOPWORDS = {
    "and",
    "for",
    "the",
    "with",
    "from",
    "that",
    "this",
    "into",
    "over",
    "under",
    "each",
    "each",
    "shall",
    "will",
    "provide",
    "support",
    "services",
    "service",
    "work",
    "includes",
    "including",
    "deliver",
    "deliverables",
    "documentation",
    "coordination",
    "contract",
    "award",
    "task",
    "tasks",
    "project",
}


@dataclass(frozen=True)
class Document:
    award_id: str
    description: str


@dataclass(frozen=True)
class Chunk:
    strategy: str
    award_id: str
    chunk_index: int
    text: str


def tokenize(text: str) -> list[str]:
    return [token.lower() for token in WORD_RE.findall(text)]


def word_count(text: str) -> int:
    return len(tokenize(text))



def normalize_vector(vector: list[float]) -> list[float]:
    norm = math.sqrt(sum(value * value for value in vector))
    if norm == 0:
        return vector
    return [value / norm for value in vector]


def cosine_similarity(left: list[float], right: list[float]) -> float:
    return sum(l_value * r_value for l_value, r_value in zip(left, right))


def run_coroutine(coro: Any) -> Any:
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    if not loop.is_running():
        return loop.run_until_complete(coro)

    result_box: dict[str, Any] = {}
    error_box: dict[str, BaseException] = {}

    def _runner() -> None:
        try:
            result_box["value"] = asyncio.run(coro)
        except BaseException as exc:  # pragma: no cover - notebook fallback
            error_box["error"] = exc

    thread = threading.Thread(target=_runner, daemon=True)
    thread.start()
    thread.join()
    if error_box:
        raise error_box["error"]
    return result_box.get("value")


def resolve_database_url() -> str | None:
    env_url = os.getenv("DATABASE_URL")
    if env_url:
        return env_url
    try:
        from govintel.config import get_settings
    except Exception:
        return None
    try:
        return get_settings().database_url
    except Exception:
        return None


data_load_error: str | None = None


async def fetch_contract_rows(limit: int = 100) -> list[dict[str, str]]:
    global data_load_error
    data_load_error = None
    database_url = resolve_database_url()
    if not database_url:
        data_load_error = "DATABASE_URL is not configured and project settings could not be loaded"
        return []

    try:
        from sqlalchemy import text
        from sqlalchemy.ext.asyncio import create_async_engine
    except Exception as exc:
        data_load_error = f"SQLAlchemy import failed: {type(exc).__name__}: {exc}"
        return []

    engine = create_async_engine(database_url, echo=False)
    try:
        async with engine.connect() as conn:
            result = await conn.execute(
                text(
                    """
                    SELECT award_id, description
                    FROM contracts
                    WHERE description IS NOT NULL AND TRIM(description) <> ''
                    ORDER BY award_id
                    LIMIT :limit
                    """
                ),
                {"limit": limit},
            )
            return [dict(row._mapping) for row in result]
    except Exception as exc:
        data_load_error = f"PostgreSQL query failed for {database_url}: {type(exc).__name__}: {exc}"
        return []
    finally:
        await engine.dispose()


def build_fallback_documents(limit: int = 100, *, start_index: int = 0) -> list[Document]:
    topics = [
        "cybersecurity incident response",
        "cloud migration and platform engineering",
        "data analytics and dashboard modernization",
        "facilities maintenance and custodial support",
        "fleet logistics and vehicle sustainment",
        "training curriculum and change management",
        "health IT integration and interoperability",
        "environmental sampling and compliance reporting",
        "field operations support and procurement planning",
        "software testing and quality assurance",
    ]
    modifiers = [
        "for headquarters and regional offices",
        "for mission support centers",
        "for secure program offices",
        "for field operations teams",
        "for shared service centers",
        "for cross-agency modernization",
        "for records and reporting teams",
        "for customer support desks",
        "for grant program managers",
        "for acquisition teams",
    ]
    deliverables = [
        "weekly status reports",
        "transition playbooks",
        "monthly metrics dashboards",
        "standard operating procedures",
        "training sessions",
        "risk logs",
        "deployment checklists",
        "quality gates",
        "acceptance summaries",
        "closeout packages",
    ]

    documents: list[Document] = []
    for offset in range(limit):
        index = start_index + offset
        topic = topics[index % len(topics)]
        modifier = modifiers[(index // len(topics)) % len(modifiers)]
        deliverable = deliverables[index % len(deliverables)]
        description = (
            f"Provide {topic} services {modifier}. "
            f"Deliver {deliverable} and maintain cycle {index:03d} artifacts for audit readiness. "
            "Support stakeholder coordination, issue triage, documentation updates, and transition planning. "
            "Track milestone dependencies, vendor touchpoints, operational risk, and monthly governance reviews."
        )
        documents.append(Document(award_id=f"sample-{index:03d}", description=description))
    return documents


def load_documents(limit: int = 100) -> tuple[list[Document], str]:
    rows = run_coroutine(fetch_contract_rows(limit=limit))
    if rows:
        documents = [
            Document(award_id=str(row.get("award_id", "")), description=str(row.get("description", "")))
            for row in rows
            if str(row.get("description", "")).strip()
        ]
        if len(documents) >= limit:
            return documents[:limit], f"postgresql({len(documents[:limit])} rows)"
        padding = build_fallback_documents(limit - len(documents), start_index=len(documents))
        return documents + padding, f"postgresql({len(documents)} rows) + fallback_padding({len(padding)} rows)"

    return build_fallback_documents(limit), "fallback_sample_data"


documents, data_source_note = load_documents(limit=100)
print(f"Loaded {len(documents)} descriptions from: {data_source_note}")
if data_load_error:
    print(f"Fallback reason: {data_load_error}")
print("Real data path active." if data_source_note.startswith("postgresql") else "Fallback sample path active.")


In [ ]:
class EmbeddingBackend:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2", fallback_dim: int = 96) -> None:
        self.model_name = model_name
        self.fallback_dim = fallback_dim
        self._real_model: Any = None
        self._real_error: BaseException | None = None

    def _load_real_model(self) -> Any:
        if self._real_model is not None or self._real_error is not None:
            return self._real_model
        if os.getenv("GOVINTEL_BENCHMARK_FORCE_FALLBACK") == "1":
            self._real_error = RuntimeError("GOVINTEL_BENCHMARK_FORCE_FALLBACK=1")
            return None
        try:
            from govintel.ingestion.embedder import EmbeddingModel

            self._real_model = EmbeddingModel(self.model_name)
        except BaseException as exc:
            self._real_error = exc
            self._real_model = None
        return self._real_model

    @property
    def mode(self) -> str:
        return "real" if self._load_real_model() is not None else "fallback"

    @property
    def fallback_reason(self) -> str | None:
        return None if self._real_error is None else str(self._real_error)

    def _hash_vector(self, text: str) -> list[float]:
        vector = [0.0] * self.fallback_dim
        tokens = tokenize(text)
        if not tokens:
            return vector
        for token in tokens:
            digest = hashlib.blake2b(token.encode("utf-8"), digest_size=16).digest()
            for slot_index in range(0, len(digest), 2):
                slot = ((digest[slot_index] << 8) | digest[slot_index + 1]) % self.fallback_dim
                vector[slot] += 1.0
        return normalize_vector(vector)

    def encode(self, texts: list[str]) -> list[list[float]]:
        real_model = self._load_real_model()
        if real_model is not None:
            try:
                return real_model.encode(texts)
            except BaseException as exc:
                self._real_error = exc
                self._real_model = None
        return [self._hash_vector(text) for text in texts]


backend = EmbeddingBackend()
print(f"Embedding backend mode: {backend.mode}")
if backend.fallback_reason:
    print(f"Fallback reason: {backend.fallback_reason}")


def build_chunks(documents: list[Document], strategy_name: str) -> list[Chunk]:
    chunks: list[Chunk] = []
    for document in documents:
        if strategy_name == "chunk_fixed":
            chunk_texts = project_chunk_fixed(document.description, size=320, overlap=60)
        elif strategy_name == "chunk_sentence":
            chunk_texts = project_chunk_sentence(document.description, max_sentences=2)
        elif strategy_name == "chunk_semantic":
            chunk_texts = project_chunk_semantic(
                document.description,
                embedding_model=backend,
                similarity_threshold=0.62,
            )
        else:
            raise ValueError(f"Unknown strategy: {strategy_name}")

        chunks.extend(
            Chunk(
                strategy=strategy_name,
                award_id=document.award_id,
                chunk_index=index,
                text=chunk_text,
            )
            for index, chunk_text in enumerate(chunk_texts)
        )
    return chunks



In [ ]:
def pick_anchor_indices(total: int, count: int = 10) -> list[int]:
    if total <= count:
        return list(range(total))
    raw_indices = [round(index * (total - 1) / (count - 1)) for index in range(count)]
    selected: list[int] = []
    seen: set[int] = set()
    for candidate in raw_indices:
        if candidate not in seen:
            selected.append(candidate)
            seen.add(candidate)
    fallback = 0
    while len(selected) < count:
        if fallback not in seen:
            selected.append(fallback)
            seen.add(fallback)
        fallback += 1
    return selected[:count]


def make_query(description: str) -> str:
    tokens = [token for token in tokenize(description) if token not in STOPWORDS and len(token) > 4]
    if not tokens:
        return "contract services"
    counts = Counter(tokens)
    ranked = sorted(set(tokens), key=lambda token: (-counts[token], -len(token), token))
    core = ranked[:4]
    if len(core) >= 3:
        return f"{core[0]} {core[1]} {core[2]}"
    return " ".join(core)


def build_queries(documents: list[Document], count: int = 10) -> list[dict[str, str]]:
    anchor_indices = pick_anchor_indices(len(documents), count=count)
    queries: list[dict[str, str]] = []
    for anchor_index in anchor_indices:
        document = documents[anchor_index]
        queries.append(
            {
                "query": make_query(document.description),
                "relevant_award_id": document.award_id,
                "anchor_index": str(anchor_index),
            }
        )
    return queries


queries = build_queries(documents, count=10)
print("Anchor queries:")
for index, query in enumerate(queries, start=1):
    print(f"{index:02d}. {query['query']}  ->  {query['relevant_award_id']}")


In [ ]:
def evaluate_strategy(strategy_name: str, documents: list[Document], queries: list[dict[str, str]]) -> dict[str, Any]:
    chunks = build_chunks(documents, strategy_name)
    chunk_texts = [chunk.text for chunk in chunks]

    embedding_start = time.perf_counter()
    chunk_vectors = backend.encode(chunk_texts)
    embedding_time_s = time.perf_counter() - embedding_start

    chunk_lengths = [word_count(chunk.text) for chunk in chunks]
    results: list[dict[str, float]] = []

    for query in queries:
        query_vector = backend.encode([query["query"]])[0]
        scored = sorted(
            ((cosine_similarity(query_vector, vector), chunk) for vector, chunk in zip(chunk_vectors, chunks)),
            key=lambda item: item[0],
            reverse=True,
        )[:5]
        relevant_chunk_ids = {
            f"{chunk.award_id}:{chunk.chunk_index}"
            for chunk in chunks
            if chunk.award_id == query["relevant_award_id"]
        }
        hits = sum(
            1
            for _, chunk in scored
            if f"{chunk.award_id}:{chunk.chunk_index}" in relevant_chunk_ids
        )
        precision_at_5 = hits / 5
        recall_at_5 = hits / max(len(relevant_chunk_ids), 1)
        results.append({"precision_at_5": precision_at_5, "recall_at_5": recall_at_5})

    avg_precision = statistics.mean(item["precision_at_5"] for item in results) if results else 0.0
    avg_recall = statistics.mean(item["recall_at_5"] for item in results) if results else 0.0
    avg_chunk_length = statistics.mean(chunk_lengths) if chunk_lengths else 0.0

    return {
        "strategy": strategy_name,
        "chunk_count": len(chunks),
        "avg_chunk_length_words": avg_chunk_length,
        "embedding_time_s": embedding_time_s,
        "precision_at_5": avg_precision,
        "recall_at_5": avg_recall,
    }


strategies = ["chunk_fixed", "chunk_sentence", "chunk_semantic"]
comparison_rows = [evaluate_strategy(strategy, documents, queries) for strategy in strategies]

def f1_score(precision: float, recall: float) -> float:
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

for row in comparison_rows:
    row["f1_at_5"] = f1_score(row["precision_at_5"], row["recall_at_5"])

comparison_rows


In [ ]:
def format_number(value: float, digits: int = 3) -> str:
    return f"{value:.{digits}f}"


def render_markdown_table(rows: list[dict[str, Any]]) -> str:
    headers = [
        "strategy",
        "chunk_count",
        "avg_chunk_length_words",
        "embedding_time_s",
        "precision_at_5",
        "recall_at_5",
        "f1_at_5",
    ]
    header_row = "| " + " | ".join(headers) + " |"
    separator_row = "| " + " | ".join("---" for _ in headers) + " |"
    body_rows = []
    for row in rows:
        body_rows.append(
            "| "
            + " | ".join(
                [
                    str(row["strategy"]),
                    str(row["chunk_count"]),
                    format_number(float(row["avg_chunk_length_words"]), 1),
                    format_number(float(row["embedding_time_s"])),
                    format_number(float(row["precision_at_5"])),
                    format_number(float(row["recall_at_5"])),
                    format_number(float(row["f1_at_5"])),
                ]
            )
            + " |"
        )
    return "\n".join([header_row, separator_row, *body_rows])


comparison_rows = sorted(
    comparison_rows,
    key=lambda row: (
        -float(row["f1_at_5"]),
        -float(row["recall_at_5"]),
        -float(row["precision_at_5"]),
        float(row["embedding_time_s"]),
    ),
)

print("Comparison table:\n")
print(render_markdown_table(comparison_rows))

best_row = comparison_rows[0]
recommendation = (
    f"Recommend {best_row['strategy']} because it has the strongest combined precision@5 and recall@5 "
    f"for this corpus, with {best_row['chunk_count']} chunks, average chunk length "
    f"{format_number(float(best_row['avg_chunk_length_words']), 1)} words, and embedding time "
    f"{format_number(float(best_row['embedding_time_s']))} seconds."
)
print("\nRecommendation:\n")
print(recommendation)


## Notebook notes

- The PostgreSQL path is the first branch exercised by the loader. If the database connection or query fails, the notebook falls back to the deterministic sample corpus and keeps running.
- The embedding backend prefers the project `EmbeddingModel` and falls back to a deterministic hash vectorizer when that model cannot be loaded.
- The evaluation uses 10 anchor queries derived from the loaded descriptions so every query has a known relevant document even when the notebook runs without real production data.
